# Elaborazione video frame-by-frame con OpenCV

**Corso:** Trattamento di Dati Multimediali   
**Strumenti:** Python, OpenCV (`opencv-python`), NumPy, Matplotlib

---

In questa lezione costruiamo una **pipeline di elaborazione video personalizzabile**: leggiamo un file video frame per frame, applichiamo filtri di elaborazione a ciascun frame e salviamo il risultato su un nuovo file video.

Ogni frame di un video è a tutti gli effetti un'immagine. Le tecniche di elaborazione studiate nelle lezioni precedenti (manipolazione pixel, analisi dell'istogramma, conversione degli spazi colore) si applicano quindi direttamente, frame per frame, all'intero flusso video.

## 0. Importazione delle librerie

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import time

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version:  {np.__version__}")

## 1. Struttura di un video digitale

Un video digitale è una sequenza ordinata di **frame** (immagini statiche) riprodotti a una frequenza fissa chiamata **frame rate**, espressa in fotogrammi al secondo (fps).

Le principali proprietà di un video sono:

- **Risoluzione** (larghezza × altezza in pixel): definisce la dimensione spaziale di ogni frame.
- **Frame rate** (fps): definisce la fluidità del movimento. Valori tipici: 24 fps (cinema), 25/30 fps (broadcast), 60 fps (gaming/sport).
- **Durata** (in secondi): ricavabile come `numero_totale_frame / fps`.
- **Codec**: algoritmo usato per comprimere e decomprimere il flusso video (es. H.264/AVC, H.265/HEVC, VP9).

In OpenCV, l'oggetto `cv2.VideoCapture` permette di aprire un file video (o un dispositivo di acquisizione) e accedere ai suoi frame uno alla volta, come se si scorresse una lista di immagini.

## 2. Lettura e ispezione di un file video

### 2.1 Aprire il video e leggere i metadati

`VideoCapture` espone alcune proprietà accessibili tramite il metodo `.get()`, usando le costanti `cv2.CAP_PROP_*`.

In [ ]:
VIDEO_PATH = "sample2.mp4"   # sostituire con il percorso del proprio file video

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Impossibile aprire il file: {VIDEO_PATH}")

fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration     = total_frames / fps if fps > 0 else 0

print(f"Risoluzione    : {width} x {height} px")
print(f"Frame rate     : {fps:.2f} fps")
print(f"Frame totali   : {total_frames}")
print(f"Durata stimata : {duration:.2f} s")

cap.release()

### 2.2 Lettura e visualizzazione del primo frame

OpenCV legge i frame in formato **BGR** (Blue-Green-Red). Prima di visualizzare un frame con Matplotlib è necessario invertire l'ordine dei canali con `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)`.

Questo aspetto è già noto dalle lezioni precedenti; lo richiamiamo brevemente perché in questa lezione lavoriamo esclusivamente con OpenCV.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()   # ret=True se la lettura ha avuto successo

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 4.5))
    plt.imshow(frame_rgb)
    plt.title("Frame 0 (primo frame)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Lettura del primo frame fallita.")

cap.release()

### 2.3 Estrazione di frame a intervalli regolari

Una tecnica comune per avere una panoramica del contenuto di un video è estrarre un frame ogni `N` secondi e visualizzarli in griglia.

**Nota su `cap.release()`**

La chiamata `cap.release()` dice a OpenCV di chiudere il file video e liberare le risorse associate: l'handle al file, i buffer interni e il decoder. Senza di essa il file rimane aperto finché il garbage collector di Python non distrugge l'oggetto, il che può avvenire in momenti imprevedibili.

Nel contesto di un notebook, dove le celle vengono rieseguite frequentemente, omettere `cap.release()` può causare:

- **file bloccato**: su Windows un file video aperto non può essere sovrascritto o spostato da altri processi;
- **accumulo di handle**: ogni riesecuzione della cella apre un nuovo `VideoCapture` sullo stesso file senza chiudere il precedente;
- **output corrotto**: su `VideoWriter`, il trailer del container (che contiene la durata totale e altri metadati) viene scritto solo al momento del rilascio — omettere `out.release()` produce un file video non riproducibile correttamente.

Il pattern corretto in Python sarebbe un context manager (`with cv2.VideoCapture(...) as cap`), ma OpenCV non supporta nativamente il protocollo `with`. La chiamata esplicita a `.release()` è quindi la forma standard.

In [ ]:
INTERVALLO_S = 2   # estrai un frame ogni 2 secondi

cap = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS)
step = max(1, int(fps * INTERVALLO_S))

thumbnails = []
frame_idx  = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % step == 0:
        thumbnails.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_idx += 1

cap.release()
print(f"Frame estratti: {len(thumbnails)}")

# Visualizzazione in griglia
n_cols = 4
n_rows = max(1, (len(thumbnails) + n_cols - 1) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = np.array(axes).flatten()

for i, ax in enumerate(axes):
    if i < len(thumbnails):
        ax.imshow(thumbnails[i])
        ax.set_title(f"t = {i * INTERVALLO_S} s", fontsize=9)
    ax.axis("off")

plt.suptitle("Thumbnail estratti dal video", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Applicazione di filtri frame-by-frame

In questa sezione definiamo quattro filtri di elaborazione e costruiamo una funzione `apply_filter()` che li rende intercambiabili. Ogni filtro è presentato nella propria cella markdown prima del codice.

### 3.1 Conversione in scala di grigi

La conversione in scala di grigi elimina l'informazione cromatica e conserva solo la **luminanza**. OpenCV usa la formula pesata ITU-R BT.601, già incontrata nella Lezione 2:

$$Y = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$

Il risultato è un'immagine a **singolo canale**. Per salvarlo in un video (che richiede immagini a 3 canali) è necessario riconvertirlo con `cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)`.

In [ ]:
def filtro_grayscale(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

### 3.2 Sfocatura gaussiana

La sfocatura gaussiana applica una **convoluzione** con un kernel gaussiano 2D a ciascun canale dell'immagine. Ogni pixel del risultato è una media pesata dei pixel vicini, con pesi decrescenti allontanandosi dal centro (distribuzione di Gauss).

Parametri principali:
- `ksize`: dimensione del kernel (deve essere dispari, es. `(15, 15)`). Kernel più grandi producono una sfocatura più marcata.
- `sigmaX`: deviazione standard lungo l'asse X. Se impostato a 0, OpenCV lo calcola automaticamente dalla dimensione del kernel.

La sfocatura gaussiana è spesso usata come passo di pre-processing per ridurre il rumore prima di filtri più sensibili (es. Canny).

In [ ]:
def filtro_blur(frame_bgr, ksize=(15, 15)):
    return cv2.GaussianBlur(frame_bgr, ksize, sigmaX=0)

### 3.3 Rilevamento dei bordi con Canny

L'algoritmo di Canny individua i **bordi** dell'immagine, ossia le zone in cui l'intensità varia bruscamente. Le fasi principali sono:

1. Sfocatura gaussiana (riduzione del rumore).
2. Calcolo del gradiente dell'intensità (operatori di Sobel).
3. Non-maximum suppression: si conservano solo i massimi locali lungo la direzione del gradiente.
4. Double thresholding: i pixel vengono classificati come "forti" (> `threshold2`), "deboli" (`threshold1` < p < `threshold2`) o da scartare.
5. Edge tracking by hysteresis: i pixel deboli adiacenti a pixel forti vengono accettati come bordi.

I parametri `threshold1` e `threshold2` controllano la sensibilità del rilevatore. Il risultato è un'immagine binaria a singolo canale.

In [ ]:
def filtro_canny(frame_bgr, threshold1=50, threshold2=150):
    gray  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, threshold1, threshold2)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

### 3.4 Sogliatura adattiva

Nella sogliatura globale si confronta l'intensità di ogni pixel con una soglia fissa valida per tutta l'immagine. Questo approccio fallisce in presenza di illuminazione non uniforme: zone in ombra o sovraesposte verrebbero classificate erroneamente.

La **sogliatura adattiva** risolve il problema calcolando una soglia locale per ogni pixel. Il procedimento è il seguente:

1. Per ogni pixel si considera un intorno quadrato di dimensione `blockSize × blockSize`.
2. Si calcola la media (o la media gaussiana pesata) delle intensità dei pixel nell'intorno.
3. Si sottrae la costante `C` a tale media: il risultato è la soglia locale per quel pixel.
4. Il pixel viene impostato a **bianco** (255) se la sua intensità è maggiore della soglia locale, a **nero** (0) altrimenti.

Il risultato è un'immagine binaria in cui la decisione per ogni pixel dipende esclusivamente dal suo contesto locale, indipendentemente dall'illuminazione globale.

In [ ]:
def filtro_adaptive_threshold(frame_bgr, blockSize=11, C=2):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(
        gray,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY,
        blockSize=blockSize,
        C=C
    )
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

### 3.5 Funzione dispatcher `apply_filter`

Raccogliamo i quattro filtri in un'unica funzione che accetta un parametro `mode` per selezionare il filtro desiderato.

In [ ]:
FILTRI_DISPONIBILI = ["grayscale", "blur", "canny", "adaptive_threshold"]

def apply_filter(frame_bgr, mode="grayscale"):
    if mode == "grayscale":
        return filtro_grayscale(frame_bgr)
    elif mode == "blur":
        return filtro_blur(frame_bgr)
    elif mode == "canny":
        return filtro_canny(frame_bgr)
    elif mode == "adaptive_threshold":
        return filtro_adaptive_threshold(frame_bgr)
    else:
        raise ValueError(f"Filtro '{mode}' non riconosciuto. Scegliere tra: {FILTRI_DISPONIBILI}")

### 3.6 Preview dei filtri su un frame campione

Prima di processare l'intero video, verifichiamo visivamente l'effetto di ciascun filtro su un frame rappresentativo.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)  # posizionati sul frame centrale
ret, frame_campione = cap.read()
cap.release()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

titoli = ["Originale"] + FILTRI_DISPONIBILI
frames = [cv2.cvtColor(frame_campione, cv2.COLOR_BGR2RGB)] + [
    cv2.cvtColor(apply_filter(frame_campione, m), cv2.COLOR_BGR2RGB)
    for m in FILTRI_DISPONIBILI
]

for ax, img, titolo in zip(axes, frames, titoli):
    ax.imshow(img)
    ax.set_title(titolo, fontsize=10)
    ax.axis("off")

plt.suptitle("Confronto filtri su frame campione", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Pipeline di processing e salvataggio del video

### 4.1 Funzione `process_video`

La funzione legge il video di input frame per frame, applica il filtro scelto e scrive il risultato in un nuovo file video tramite `cv2.VideoWriter`.

`VideoWriter` richiede:
- Il **path** del file di output.
- Un **fourcc** (Four-Character Code) che identifica il codec: su macOS si usa `avc1` (H.264); su Linux è preferibile `mp4v`.
- Il **frame rate** dell'output (tipicamente uguale all'input).
- La **risoluzione** (larghezza, altezza) dei frame di output.

**Nota sul codec su macOS**

Su macOS, `cv2.VideoWriter` con il codec `mp4v` e l'estensione `.mp4` spesso fallisce silenziosamente: crea il file ma non ci scrive nulla, oppure produce un file vuoto. Il problema è che OpenCV non solleva eccezioni in questo caso — `out.isOpened()` può restituire `True` anche quando la scrittura non avverrà.

La soluzione è usare il codec `avc1` (H.264 nativo su macOS):

```python
fourcc = cv2.VideoWriter_fourcc(*"avc1")
```

Per intercettare il fallimento prima di eseguire l'intero loop, si aggiunge un controllo esplicito su `out.isOpened()` subito dopo la creazione del `VideoWriter`:

```python
if not out.isOpened():
    cap.release()
    raise RuntimeError("VideoWriter non riuscito ad aprire il file di output.")
```

Senza questo controllo, il loop verrebbe completato integralmente senza produrre nulla, e l'errore sarebbe scoperto solo alla fine.

In [ ]:
def process_video(input_path, output_path, codec="avc1", mode="grayscale", verbose=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    if os.path.exists(output_path):
        os.remove(output_path)

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*codec)   # su Linux usare 'mp4v'
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Prova a cambiare codec (avc1 su macOS, mp4v su Linux) "
            "o l'estensione del file."
        )

    t_start = time.perf_counter()
    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        filtered = apply_filter(frame, mode)
        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()

    elapsed    = time.perf_counter() - t_start
    throughput = n_frame / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"Filtro         : {mode}")
        print(f"Frame scritti  : {n_frame} / {total}")
        print(f"Tempo totale   : {elapsed:.2f} s")
        print(f"Throughput     : {throughput:.1f} frame/s")
        print(f"Output salvato : {output_path}")

    return n_frame, elapsed

### 4.2 Esecuzione della pipeline

Modifica la variabile `FILTRO_SCELTO` per selezionare il filtro da applicare.

In [ ]:
FILTRO_SCELTO = "canny"   # scegliere tra: grayscale, blur, canny, adaptive_threshold

output_path = f"output_{FILTRO_SCELTO}.mp4"
n_frame, elapsed = process_video(VIDEO_PATH, output_path, mode=FILTRO_SCELTO)

### 4.3 Confronto visivo: frame originale vs. frame processato

Estraiamo il frame centrale dal video di output e lo affianchiamo al corrispondente frame originale.

In [ ]:
def leggi_frame_n(path, n):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, n)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

frame_centrale = n_frame // 2
frame_orig = leggi_frame_n(VIDEO_PATH,  frame_centrale)
frame_proc = leggi_frame_n(output_path, frame_centrale)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame_orig, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Originale — frame {frame_centrale}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(frame_proc, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Filtro: {FILTRO_SCELTO} — frame {frame_centrale}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Pipeline personalizzabile con filtri in cascata

Fino a questo punto ogni video veniva processato con un solo filtro. È possibile comporre più filtri in sequenza: l'output di ogni filtro diventa l'input del successivo.

### 5.1 Definizione della pipeline

In [ ]:
def apply_pipeline(frame_bgr, pipeline):
    """Applica in sequenza una lista di filtri al frame."""
    result = frame_bgr.copy()
    for mode in pipeline:
        result = apply_filter(result, mode)
    return result


def process_video_pipeline(input_path, output_path, pipeline, verbose=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossibile aprire: {input_path}")

    if os.path.exists(output_path):
        os.remove(output_path)

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"avc1")   # su Linux usare 'mp4v'
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Prova a cambiare codec (avc1 su macOS, mp4v su Linux) "
            "o l'estensione del file."
        )

    t_start = time.perf_counter()
    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        filtered = apply_pipeline(frame, pipeline)
        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()

    elapsed    = time.perf_counter() - t_start
    throughput = n_frame / elapsed if elapsed > 0 else 0

    if verbose:
        print(f"Pipeline       : {' -> '.join(pipeline)}")
        print(f"Frame scritti  : {n_frame} / {total}")
        print(f"Tempo totale   : {elapsed:.2f} s")
        print(f"Throughput     : {throughput:.1f} frame/s")
        print(f"Output salvato : {output_path}")

    return n_frame, elapsed

### 5.2 Esecuzione della pipeline personalizzata

Modifica la lista `MIA_PIPELINE` aggiungendo, rimuovendo o riordinando i filtri.

> **Attenzione alla compatibilità tra filtri**: `adaptive_threshold` e `canny` restituiscono immagini binarie. Applicare `blur` prima di essi riduce il rumore prima della sogliatura; applicarlo dopo attenua i contorni. Osserva come cambia il risultato al variare dell'ordine.

In [ ]:
MIA_PIPELINE = ["blur", "canny"]   # personalizza la sequenza di filtri

output_pipeline = "output_pipeline.mp4"
process_video_pipeline(VIDEO_PATH, output_pipeline, pipeline=MIA_PIPELINE)

### 5.3 Confronto visivo dei stadi della pipeline

Visualizziamo l'effetto di ogni stadio della pipeline su un frame campione.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)  #
ret, frame_test = cap.read()
cap.release()

stadi     = [frame_test]
etichette = ["Originale"]
corrente  = frame_test.copy()

for filtro in MIA_PIPELINE:
    corrente = apply_filter(corrente, filtro)
    stadi.append(corrente.copy())
    etichette.append(filtro)

fig, axes = plt.subplots(1, len(stadi), figsize=(5 * len(stadi), 4))
for ax, img, label in zip(axes, stadi, etichette):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=10)
    ax.axis("off")

plt.suptitle(f"Stadi della pipeline: {' -> '.join(etichette)}", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Sfida opzionale — Overlay di testo sui frame

`cv2.putText` permette di sovrapporre testo a ogni frame prima della scrittura sul video di output. È utile per aggiungere il numero del frame, il timestamp o etichette diagnostiche.

**Parametri principali di `cv2.putText`:**
- `img`: frame su cui scrivere (modificato in-place).
- `text`: stringa da visualizzare.
- `org`: coordinate (x, y) dell'angolo in basso a sinistra del testo.
- `fontFace`: stile del font (es. `cv2.FONT_HERSHEY_SIMPLEX`).
- `fontScale`: fattore di scala del font.
- `color`: colore in BGR.
- `thickness`: spessore del tratto in pixel.
- `lineType`: `cv2.LINE_AA` per anti-aliasing (testo più pulito).

In [ ]:
def process_video_con_overlay(input_path, output_path, mode="grayscale"):
    cap    = cv2.VideoCapture(input_path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if os.path.exists(output_path):
        os.remove(output_path)

    fourcc = cv2.VideoWriter_fourcc(*"avc1")   # su Linux usare 'mp4v'
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        cap.release()
        raise RuntimeError(
            f"VideoWriter non riuscito ad aprire '{output_path}'. "
            "Prova a cambiare codec (avc1 su macOS, mp4v su Linux) "
            "o l'estensione del file."
        )

    n_frame = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        filtered  = apply_filter(frame, mode)
        timestamp = n_frame / fps
        testo     = f"Frame: {n_frame:04d}  |  t = {timestamp:.2f} s  |  filtro: {mode}"

        cv2.putText(
            filtered, testo,
            org=(10, height - 15),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=0.45,
            color=(200, 200, 200),
            thickness=1,
            lineType=cv2.LINE_AA
        )

        out.write(filtered)
        n_frame += 1

    cap.release()
    out.release()
    print(f"Video con overlay salvato: {output_path} ({n_frame} frame)")
    return n_frame


n_overlay = process_video_con_overlay(VIDEO_PATH, "output_overlay.mp4", mode="grayscale")

# Verifica su un frame campione
frame_ov = leggi_frame_n("output_overlay.mp4", n_overlay // 2)
if frame_ov is not None:
    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(frame_ov, cv2.COLOR_BGR2RGB))
    plt.title("Frame con overlay di testo")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Confronto del throughput tra filtri

Il costo computazionale dei filtri varia significativamente. Misuriamo il throughput (frame/secondo) per ciascuno dei quattro filtri sullo stesso video e confrontiamo i risultati.

I valori ottenuti dipendono dall'hardware, dal sistema operativo e soprattutto dalla risoluzione del video utilizzato: aumentare la risoluzione quadruplica (o più) il numero di pixel per frame e riduce il throughput proporzionalmente.

In [ ]:
risultati = {}

for filtro in FILTRI_DISPONIBILI:
    out_tmp = f"_tmp_{filtro}.mp4"
    n, elapsed = process_video(VIDEO_PATH, out_tmp, mode=filtro, verbose=False)
    throughput = n / elapsed if elapsed > 0 else 0
    risultati[filtro] = throughput
    print(f"{filtro:<22} : {throughput:6.1f} frame/s")
    if os.path.exists(out_tmp):
        os.remove(out_tmp)

# Grafico a barre
fig, ax = plt.subplots(figsize=(8, 4))
filtri = list(risultati.keys())
valori = list(risultati.values())
colori = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
bars   = ax.bar(filtri, valori, color=colori)
ax.set_ylabel("Throughput (frame/s)")
ax.set_title("Throughput per filtro\n(i valori dipendono dall'hardware e dalla risoluzione del video)")
for bar, v in zip(bars, valori):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Riepilogo

In questa lezione abbiamo costruito una pipeline di elaborazione video frame-by-frame con OpenCV:

- **Lettura e ispezione**: `cv2.VideoCapture` per accedere ai metadati (fps, risoluzione, durata) e ai singoli frame.
- **Filtri frame-by-frame**: quattro trasformazioni — scala di grigi, sfocatura gaussiana, rilevamento bordi (Canny), sogliatura adattiva.
- **Scrittura**: `cv2.VideoWriter` per salvare il video processato con codec e frame rate configurabili.
- **Pipeline in cascata**: composizione di più filtri in sequenza tramite `apply_pipeline`.
- **Overlay**: aggiunta di informazioni testuali su ogni frame con `cv2.putText`.
- **Throughput**: misurazione empirica del costo computazionale di ciascun filtro.